# DBMS

目录:
- DBMS的体系结构
    - 内核组成
    - 进程结构
- 数据库权限管理
- 查询优化
- 事务管理
    - 并发控制 
    - 数据恢复

## 数据库内核接收到一条sql之后的处理过程

- sql解析 -> 权限检查 -> 语法分析和查询处理 -> (并发控制/Acceess Management/恢复机制) -> 调用操作系统接口 
- 访问原语是数据库最底层API


## DBMS的进程结构

- 单进程
- 多进程
- 多线程

### 单进程

类似foxpro,将数据库和DBMS捆绑一起

### 多进程 Process

企业级应用,多用户/多任务:
- DBMS核心单独进程
- 应用程序代码单独进程
- 每一个需要访问数据库的应用,系统会生成一个DBMS核心进程为其服务


> CONNECT命令  
> 创建一个DBMS核心进程,同时为应用进程和DBMS核心进程构建一个`管道`  
> 单机环境下用: 管道(Pipe),单向通信机制,读写管道    
> 联网环境下用: socket  


### 多线程 Thread

- 只有一个DBMS进程
- 进程空间会为每一个需要访问数据库的应用创建一个线程

> daemon 守护进程  
> 负责监听 数据库接收连接请求的IP地址和端口号  
> Catalog 目录  
> 数据的数据,元数据   
> lock table 封锁表 
> 数据库的并发控制  
> buffer 缓冲区 

应用程序访问数据库的流程:
- 调用CONNECT,DAEMON捕获请求
- 在DBMS的进程空间创建一个`核心线程`
- 并为应用程序进程和`核心线程`之间连接`管道\socket`

## 数据库的底层实现(接近OS)

数据库系统访问类型分析:
- 查询访问涉及,大于15%元组: 堆文件
- 查找一条特定的元组: hahs文件
- 查询访问涉及,小于15%的元组
- 范围查询
- 更新

文件组织方式:
1. 堆文件是最简单的文件组织方式,随着数据插入往后添加.
2. hash文件
3. 堆文件+`B+树`索引提高数据访问效率
4. 动态hash
5. 栅格结构文件(多维数组)
6. 簇级管理:raw disk 更加底层的方式 **真实数据库软件**

> 堆文件+`B+树`索引  
> 实际的文件存储用堆文件  
> 额外针对数据文件创建一个索引文件(B+树)  

总结: 堆文件+`B+树`索引技术
- 支持大于15%的全量遍历查询 -> 使用堆作为底层
- 支持特定查询,使用了索引技术
- 支持范围查询,B+树的叶子连接

> Raw Disk  
> 磁盘数据管理机制  
> 有用户控制数据的磁盘存储方式   
> 自己实现一套文件管理  
 

索引技术:
- B+ 树
- 簇级(cluster)索引:使用rawdisk
- 倒排文件 FAT,文件检索
- 动态hash
- 栅格文件
- 位图索引


## 查询优化

- 代数优化 :  重写query
- 操作优化

代数优化:减小二元操作的数量

### 查询树

- 叶子节点表示关系
- 中间节点表示操作
- 从下而上进行计算

一组关系代数的等价变换规则示例:
- 交换律:笛卡尔乘积左右子树可交换$\Join$,$E1\Join E2 = E2\Join E1$
- 结合律:查询树可旋转,调整连接顺序$E1\Join E2 \Join E3 =E1\Join (E2 \Join E3)$
- 投影操作的串接律: 
    $\prod_{A_1\cdots A_n}(\prod_{B_1\cdots B_m(E)})=\prod_{A_1\cdots A_n(E)}$
    要求A是B的子集   
- 选择操作的串接律: $\sigma_{F1}(\sigma_{F2}(E))=\sigma_{F1\cap F2}(E)$
- 选择与投影的交换律: $\sigma_F(\prod_{A_1\cdots A_n(E)})=\prod_{A_1\cdots A_n}\sigma_F(E)$
- 如果选择F只出现E1中的属性 
    - $\sigma_F(E_1\Join E_2) = \sigma_F(E_1)\Join E_2 $,把选择操作下压,
    - $\sigma_F(E_1\Join E_2) = \sigma_{F_1}(E_1)\Join \sigma_{F_2}E_2 $,将操作尽量直接作用在关联数据上
    - $\sigma_F(E_1\Join E_2) = \sigma_{F_2}(\sigma_{F_1}(E_1)\Join E_2)$
    - $\sigma_F(E_1\cup E_2) = \sigma_{F}(E_1)\cup \sigma_{F}E_2 $
    - $\sigma_F(E_1- E_2) = \sigma_{F}(E_1) - \sigma_{F}E_2 $
 
- 投影操作也可以类似操作


### 代数优化的基本原则

1. 利等价变换规则,尽可能下压一元操作,以减少连接操作规模
2. 寻找合并公共子表达式

## 操作优化

- 优化选择操作
- 优化投影操作
- 优化集合操作
- 优化连接操作
- 优化组合操作

连接操作实现的经典算法:
- 嵌套循环,内外循环
- 优化嵌套循环,的结合物理存储,双缓存区,减少读盘次数
- 归并扫描(Merge scan),参与连接的两个关系在磁盘上实现做好了外排序,这样只需要双指针分别走一遍即可o(n+m)
- 使用B+树索引或者hash 来查找映射元组 (**最常用**),拥有索引可以加速内循环

## 恢复

企业级数据库必须具备的功能
- 减少系统故障的可能
- 错误恢复

措施:
- 冗余是必须的
- 能够检测所有可能的故障
- 周期性转储
- 备份+增量转储:可能会丢失增量
- 备份+日志 **常规方法**,通过重新演绎日志进行备份

## Transaction 事务

事务:一组对数据库进行操作的集合
- 原子性
- 一致性
- 隔离性
- 持久性

一个事务执行完毕之后就可以将结果开放给所有用户


恢复时使用的重要数据结构:
- commit list 提交事务列表 已经成功的事务列表
- active list 运行事务列表
- log 日志

两个规则:
- commit rule 提交规则,提交阶段的事务,需要将改变写到日志或者磁盘中
- 日志优先原则

## 并发控制

并发下如何保证数据的有效性
- 写写冲突
- 读写冲突

可串行化: n个并发运行的事务,随机调度,结果一致

### 封锁发并发控制

key idea: 对数据进行读写操作需要先申请锁

- x locks (排他锁) 无论是读操作或者是写操作都申请x锁
- 2 pl 两段加锁协议: 在一个事务里面,所有的加锁请求,都在锁释放前
- (S,U,X) 锁 更加精细的控制发v

> 多米诺效益; 由于一个事务的故障导致连续的错误

### 死锁问题

多个并发运行的事务出现循环等待:
- 互斥
- 循环等待
- 不可剥夺
- 请求与保持

解决方法:
- 防止死锁产生
- 发现处理死锁

> todo 暂时了解,按需填充